# Financial Knowledge Graph & LLM Stack — Complete Tutorial (ch01–ch17)

**A comprehensive, chapter-by-chapter walkthrough of building a production-grade Financial Knowledge Graph
using Neo4j, Neosemantics (n10s), APOC, FIBO ontology (TTL/RDF), and LLMs.**

This notebook covers the entire stack from the implementation plan v2:

| Phase | Chapters | Scope |
|-------|----------|-------|
| **Foundation** | ch01–ch05 | Platform, prompts, ontology, multi-source graph, entity resolution |
| **Information Extraction** | ch06–ch10 | NLP, embeddings, LLM extraction, NED, benchmarking |
| **Graph ML** | ch11–ch14 | Node2Vec, features, GNN foundations, classification & link prediction |
| **Applications** | ch15–ch17 | Graph RAG, governance, investigative Streamlit app |

---

## Prerequisites
- Neo4j 4.4+ running locally (`bolt://localhost:7687`)
- Neo4j plugins: **Neosemantics (n10s)**, **APOC**, **GDS** (optional)
- Python 3.10+
- Ollama (for local LLM) or OpenAI API key

## Architecture

```
Layer 5  App  │ Streamlit Investigator + Graph RAG    │ ch15, ch17
Layer 4  ML   │ GNNs, embeddings, feature store       │ ch11–ch14
Layer 3  IE   │ NER, NED, LLM extraction, ontology    │ ch06, ch08–ch10
Layer 2  KG   │ Canonical Financial KG (Neo4j)         │ ch03–ch05
Layer 1  Data │ Filings, news, GLEIF, FIBO             │ raw + cache
Layer 0  Plat │ util/, providers, secrets, observability│ shared
```

---
# PART 1 — ch01_fin: Platform Foundation

Establish the shared platform layer: Neo4j connection, LLM provider,
canonical schema, and configuration.

In [2]:
# ── ch01: Setup & Imports ──
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../..'))

# Platform imports
from ChaptersFinancial._platform.providers.graph import GraphProvider
from ChaptersFinancial._platform.providers.llm import LLMProvider
from ChaptersFinancial._platform.fin_importer_base import FinImporterBase

# Connect to Neo4j
gp = GraphProvider()
print('Neo4j connected:', gp.ping())

Connecting to Neo4j at... neo4j://127.0.0.1:7687...
Neo4j connected: True


In [3]:
# ── ch01: Canonical Schema & Constraints ──
# Apply all constraints from _platform/schema/constraints.cypher
from pathlib import Path
import yaml

schema_path = Path('..') / 'ChaptersFinancial' / '_platform' / 'config' / 'schema_config.yaml'
if not schema_path.exists():
    schema_path = Path('.') / '_platform' / 'config' / 'schema_config.yaml'

with open(schema_path) as f:
    schema = yaml.safe_load(f)

print('Node types:', list(schema['nodes'].keys()))
print('Relationship types:', list(schema['relationships'].keys()))

Node types: ['LegalEntity', 'Instrument', 'Exchange', 'Filing', 'StatementItem', 'Document', 'Chunk', 'Mention', 'Event', 'Transaction', 'OntologyClass', 'Run']
Relationship types: ['ISSUES', 'LISTED_ON', 'PARENT_OF', 'OWNS', 'CONTROLS', 'REPORTS_ON', 'FROM_FILING', 'OF_DOC', 'IN_CHUNK', 'MENTIONS', 'RESOLVED_TO', 'AFFECTS', 'CLASSIFIED_AS', 'EXPOSED_TO']


In [4]:
# ── ch01: Apply Schema Constraints ──
constraints = [
    'CREATE CONSTRAINT legal_entity_lei IF NOT EXISTS FOR (le:LegalEntity) REQUIRE le.lei IS UNIQUE',
    'CREATE CONSTRAINT instrument_figi IF NOT EXISTS FOR (i:Instrument) REQUIRE i.figi IS UNIQUE',
    'CREATE CONSTRAINT exchange_mic IF NOT EXISTS FOR (e:Exchange) REQUIRE e.mic IS UNIQUE',
    'CREATE CONSTRAINT document_docid IF NOT EXISTS FOR (d:Document) REQUIRE d.docId IS UNIQUE',
    'CREATE CONSTRAINT filing_filingid IF NOT EXISTS FOR (f:Filing) REQUIRE f.filingId IS UNIQUE',
    'CREATE CONSTRAINT event_eventid IF NOT EXISTS FOR (e:Event) REQUIRE e.eventId IS UNIQUE',
    'CREATE CONSTRAINT ontology_iri IF NOT EXISTS FOR (oc:OntologyClass) REQUIRE oc.iri IS UNIQUE',
    'CREATE INDEX le_name_idx IF NOT EXISTS FOR (le:LegalEntity) ON (le.name)',
    'CREATE INDEX le_cik_idx IF NOT EXISTS FOR (le:LegalEntity) ON (le.cik)',
    'CREATE INDEX instrument_ticker_idx IF NOT EXISTS FOR (i:Instrument) ON (i.ticker)',
]
for stmt in constraints:
    try:
        gp.run(stmt)
        print(f'  OK: {stmt[:60]}…')
    except Exception as e:
        print(f'  SKIP: {e}')

  OK: CREATE CONSTRAINT legal_entity_lei IF NOT EXISTS FOR (le:Leg…
  OK: CREATE CONSTRAINT instrument_figi IF NOT EXISTS FOR (i:Instr…
  OK: CREATE CONSTRAINT exchange_mic IF NOT EXISTS FOR (e:Exchange…
  OK: CREATE CONSTRAINT document_docid IF NOT EXISTS FOR (d:Docume…
  OK: CREATE CONSTRAINT filing_filingid IF NOT EXISTS FOR (f:Filin…
  OK: CREATE CONSTRAINT event_eventid IF NOT EXISTS FOR (e:Event) …
  OK: CREATE CONSTRAINT ontology_iri IF NOT EXISTS FOR (oc:Ontolog…
  OK: CREATE INDEX le_name_idx IF NOT EXISTS FOR (le:LegalEntity) …
  OK: CREATE INDEX le_cik_idx IF NOT EXISTS FOR (le:LegalEntity) O…
  OK: CREATE INDEX instrument_ticker_idx IF NOT EXISTS FOR (i:Inst…


---
# PART 2 — ch02_fin: Financial Prompting Foundations

Establish the prompting style, JSON schemas, and safety/compliance guardrails
used by every later LLM-driven chapter.

In [5]:
# ── ch02: Load JSON Schemas ──
import json
from pathlib import Path

schema_dir = Path('.') / 'ch02_fin' / 'schemas'
if not schema_dir.exists():
    schema_dir = Path('..') / 'ChaptersFinancial' / 'ch02_fin' / 'schemas'

for schema_file in sorted(schema_dir.glob('*.json')):
    with open(schema_file) as f:
        s = json.load(f)
    print(f'{schema_file.stem}: {list(s.get("properties", {}).keys())}')

entity_extraction.schema: ['entities', 'source_excerpt', 'model']
event_extraction.schema: ['events', 'source_excerpt', 'model']
filing_summary.schema: ['filing_type', 'period', 'issuer', 'key_facts', 'risks', 'forward_looking_statements', 'compliance_flags', 'model']


In [6]:
# ── ch02: Zero-Shot Entity Extraction Demo ──
llm = LLMProvider()  # Uses configured provider (mock/ollama/openai)

sample_text = """Apple Inc. reported Q3 2024 revenue of $85.8 billion, 
up 5% year-over-year. CEO Tim Cook highlighted strong iPhone sales 
and the Services segment reaching an all-time record. 
The company announced a $110 billion share buyback program."""

prompt = f"""Extract all financial entities from this text as JSON.
Return {{"entities": [{{"name": ..., "type": "ORG"|"PERSON"|"INSTRUMENT"|"LOCATION", "confidence": 0-1}}]}}

Text: {sample_text}"""

result = llm.complete_json(prompt)
print(json.dumps(result, indent=2))

{
  "entities": [
    {
      "name": "Apple Inc.",
      "type": "ORG",
      "confidence": 0.95
    },
    {
      "name": "$85.8 billion",
      "type": "INSTRUMENT",
      "confidence": 0.9
    },
    {
      "name": "5%",
      "type": "INSTRUMENT",
      "confidence": 0.85
    },
    {
      "name": "Tim Cook",
      "type": "PERSON",
      "confidence": 0.9
    },
    {
      "name": "$110 billion",
      "type": "INSTRUMENT",
      "confidence": 0.9
    }
  ]
}


In [7]:
# ── ch02: Compliance Guardrails ──
# Demonstrate the guardrail system from listing 04

# Input guardrails: reject prompts asking for price predictions
FORBIDDEN_PATTERNS = [
    'predict.*price', 'stock.*tip', 'buy.*sell.*recommendation',
    'guaranteed.*return', 'insider.*information'
]

import re
def check_input_safety(text: str) -> tuple[bool, str]:
    for pattern in FORBIDDEN_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return False, f'Blocked: matches forbidden pattern "{pattern}"'
    return True, 'OK'

# Test
print(check_input_safety('What are Apple\'s recent filings?'))
print(check_input_safety('Predict AAPL stock price for next month'))

(True, 'OK')
(False, 'Blocked: matches forbidden pattern "predict.*price"')


---
# PART 3 — ch03_fin: Financial Ontology Bootstrapping

Load FIBO (Financial Industry Business Ontology) via **Neosemantics (n10s)**,
GLEIF legal entities, and ISO 4217 currencies into Neo4j.

### TTL/RDF Ontology Loading with n10s
Neosemantics allows direct import of RDF/OWL/TTL ontologies into the property graph.

In [8]:
# ── ch03: Initialize Neosemantics (n10s) ──
# Configure n10s for RDF import
try:
    # Check if config exists
    r = gp.run('CALL n10s.graphconfig.show() YIELD param RETURN count(param) AS cnt')
    if r[0]['cnt'] == 0:
        gp.run("""
            CALL n10s.graphconfig.init({
                handleVocabUris: 'MAP',
                handleMultival: 'ARRAY',
                handleRDFTypes: 'LABELS_AND_NODES',
                keepLangTag: false,
                keepCustomDataTypes: false
            })
        """)
        print('n10s graph config initialized.')
    else:
        print('n10s already configured.')
    
    # Ensure Resource uniqueness constraint
    try:
        gp.run('CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS FOR (r:Resource) REQUIRE r.uri IS UNIQUE')
    except: pass
    print('n10s ready for RDF/TTL import.')
except Exception as e:
    print(f'n10s not available: {e}')
    print('Install the Neosemantics plugin: https://neo4j.com/labs/neosemantics/')

n10s already configured.
n10s ready for RDF/TTL import.


In [9]:
# ── ch03: Load FIBO Ontology via n10s ──
# Load Financial Industry Business Ontology (subset: BE, FBC, SEC, IND modules)
FIBO_MODULES = {
    'BE':  'https://spec.edmcouncil.org/fibo/ontology/BE/MetadataBE/BEDomain',
    'FBC': 'https://spec.edmcouncil.org/fibo/ontology/FBC/MetadataFBC/FBCDomain',
    'SEC': 'https://spec.edmcouncil.org/fibo/ontology/SEC/MetadataSEC/SECDomain',
    'IND': 'https://spec.edmcouncil.org/fibo/ontology/IND/MetadataIND/INDDomain',
}

for name, url in FIBO_MODULES.items():
    try:
        r = gp.run(
            'CALL n10s.rdf.import.fetch($url, "RDF/XML") '
            'YIELD terminationStatus, triplesLoaded '
            'RETURN terminationStatus, triplesLoaded',
            {'url': url}
        )
        print(f'  {name}: {r[0]["terminationStatus"]}, triples={r[0]["triplesLoaded"]}')
    except Exception as e:
        print(f'  {name}: SKIP ({e})')

  BE: OK, triples=66
  FBC: OK, triples=55
  SEC: OK, triples=59
  IND: OK, triples=61


In [10]:
# ── ch03: Relabel FIBO Resources as OntologyClass ──
r = gp.run("""
    MATCH (r:Resource)
    WHERE r.uri STARTS WITH 'https://spec.edmcouncil.org/fibo/'
    SET r:OntologyClass, r.source = 'FIBO', r.iri = r.uri,
        r.label = CASE
          WHEN r.`rdfs__label` IS NOT NULL AND size(r.`rdfs__label`) > 0
            THEN r.`rdfs__label`[0]
          ELSE split(r.uri, '/')[-1]
        END
    RETURN count(r) AS cnt
""")
print(f'FIBO OntologyClass nodes: {r[0]["cnt"]}')

FIBO OntologyClass nodes: 57


In [11]:
# ── ch03: Load GLEIF Legal Entities (Real Data) ──
import httpx, time, ssl, warnings

GLEIF_HEADERS = {'User-Agent': 'KG-LLM-INACTION/1.0', 'Accept': 'application/vnd.api+json'}

# Corporate proxy has a self-signed cert — disable SSL verification (like OpenRouter)
_ssl_ctx = ssl.create_default_context()
_ssl_ctx.check_hostname = False
_ssl_ctx.verify_mode = ssl.CERT_NONE
warnings.filterwarnings("ignore", message=".*Unverified.*")

# Fetch real LEI records for major financial jurisdictions
countries = ['US', 'GB', 'DE', 'JP', 'CH']
total_imported = 0

for country in countries:
    try:
        resp = httpx.get(
            'https://api.gleif.org/api/v1/lei-records',
            params={'filter[entity.legalAddress.country]': country, 'page[size]': '20'},
            headers=GLEIF_HEADERS, timeout=30,
            verify=False
        )
        resp.raise_for_status()
        records = resp.json().get('data', [])
        
        rows = []
        for rec in records:
            ent = rec['attributes']['entity']
            rows.append({
                'lei': rec['attributes']['lei'],
                'name': ent['legalName']['name'],
                'jurisdiction': ent.get('jurisdiction', country),
                'legalForm': ent.get('legalForm', {}).get('id', ''),
                'status': ent.get('status', 'ACTIVE'),
            })
        
        if rows:
            gp.run("""
                UNWIND $batch AS row
                MERGE (le:LegalEntity {lei: row.lei})
                SET le.name = row.name,
                    le.jurisdiction = row.jurisdiction,
                    le.legalForm = row.legalForm,
                    le.status = row.status,
                    le.ingestedAt = datetime()
            """, {'batch': rows})
            total_imported += len(rows)
            print(f'  {country}: {len(rows)} entities')
        time.sleep(0.5)
    except Exception as e:
        print(f'  {country}: SKIP ({e})')

print(f'\nTotal GLEIF entities imported: {total_imported}')


  US: 20 entities
  GB: 20 entities
  DE: 20 entities
  JP: 20 entities
  CH: 20 entities

Total GLEIF entities imported: 100


In [12]:
# ── ch03: Load ISO 4217 Currencies ──
try:
    import pycountry
    currencies = [{'code': c.alpha_3, 'name': c.name, 'numeric': getattr(c, 'numeric', '')}
                  for c in pycountry.currencies]
    gp.run("""
        UNWIND $batch AS row
        MERGE (c:Currency {code: row.code})
        SET c.name = row.name, c.numeric = row.numeric,
            c:OntologyClass, c.source = 'ISO4217', c.iri = 'iso4217:' + row.code,
            c.label = row.code + ' - ' + row.name
    """, {'batch': currencies})
    print(f'Loaded {len(currencies)} currencies.')
except ImportError:
    print('pycountry not installed. Run: pip install pycountry')

Loaded 178 currencies.


In [13]:
# ── ch03: Verify Ontology Load ──
for label in ['OntologyClass', 'LegalEntity', 'Currency']:
    cnt = gp.run(f'MATCH (n:{label}) RETURN count(n) AS cnt')[0]['cnt']
    print(f'  {label}: {cnt} nodes')

# Show FIBO class hierarchy sample
fibo_sample = gp.run("""
    MATCH (oc:OntologyClass {source: 'FIBO'})
    RETURN oc.label AS label, oc.iri AS iri
    ORDER BY oc.label LIMIT 10
""")
print('\nFIBO sample classes:')
for r in fibo_sample:
    lbl = r['label'] if isinstance(r['label'], str) else (r['label'][0] if r['label'] else '')
    print(f'  {lbl}')

  OntologyClass: 235 nodes
  LegalEntity: 155 nodes
  Currency: 178 nodes

FIBO sample classes:
  
  
  
  
  
  
  
  
  
  


---
# PART 4 — ch04_fin: Multi-Source Financial Graph + Community Analysis

Build the working graph: exchanges (ISO 10383 MIC), instruments (OpenFIGI),
sector classifications (SIC), ownership (SEC 13F). Then run community detection
and centrality analysis.

In [14]:
# ── ch04: Import Exchange Data (ISO 10383 MIC) ──
import csv, io

MIC_URL = 'https://www.iso20022.org/sites/default/files/ISO10383_MIC/ISO10383_MIC.csv'
try:
    resp = httpx.get(MIC_URL, timeout=30, follow_redirects=True, verify=False)
    resp.raise_for_status()
    reader = csv.DictReader(io.StringIO(resp.text))
    exchanges = []
    for row in reader:
        mic = row.get('MIC', '').strip()
        if mic and row.get('STATUS') == 'ACTIVE':
            exchanges.append({
                'mic': mic,
                'name': row.get('NAME-INSTITUTION DESCRIPTION', ''),
                'country': row.get('COUNTRY', ''),
                'operatingMic': row.get('OPERATING MIC', ''),
            })
    exchanges = exchanges[:200]  # Limit for demo
    
    gp.run("""
        UNWIND $batch AS row
        MERGE (e:Exchange {mic: row.mic})
        SET e.name = row.name, e.country = row.country,
            e.operatingMic = row.operatingMic
    """, {'batch': exchanges})
    print(f'Imported {len(exchanges)} exchanges.')
except Exception as e:
    print(f'Exchange import: {e}')


Imported 200 exchanges.


In [15]:
# ── ch04: Import Instruments via OpenFIGI ──
# Resolve S&P 500 tickers to FIGI identifiers
PILOT_TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'JPM', 'BAC', 'GS', 'PFE', 'IBM',
                 'JNJ', 'V', 'MA', 'UNH', 'HD', 'PG', 'DIS', 'NFLX', 'NVDA', 'META']

figi_results = []
for ticker in PILOT_TICKERS:
    try:
        resp = httpx.post(
            'https://api.openfigi.com/v3/mapping',
            json=[{'idType': 'TICKER', 'idValue': ticker, 'exchCode': 'US'}],
            headers={'Content-Type': 'application/json'},
            timeout=15,
            verify=False
        )
        resp.raise_for_status()
        data = resp.json()
        if data and data[0].get('data'):
            d = data[0]['data'][0]
            figi_results.append({
                'figi': d['figi'],
                'ticker': ticker,
                'name': d.get('name', ''),
                'exchCode': d.get('exchCode', ''),
                'assetClass': d.get('securityType', 'EQUITY'),
            })
        time.sleep(0.3)
    except Exception as e:
        print(f'  {ticker}: {e}')

if figi_results:
    gp.run("""
        UNWIND $batch AS row
        MERGE (i:Instrument {figi: row.figi})
        SET i.ticker = row.ticker, i.name = row.name,
            i.exchCode = row.exchCode, i.assetClass = row.assetClass,
            i.ingestedAt = datetime()
    """, {'batch': figi_results})
    
    # Link to exchanges
    gp.run("""
        MATCH (i:Instrument) WHERE i.exchCode IS NOT NULL
        MATCH (e:Exchange) WHERE e.mic = i.exchCode OR e.operatingMic = i.exchCode
        MERGE (i)-[:LISTED_ON]->(e)
    """)
    print(f'Imported {len(figi_results)} instruments via OpenFIGI.')
else:
    print('No instruments imported (API may be rate-limited).')


Imported 20 instruments via OpenFIGI.


In [16]:
# ── ch04: Import SEC 13F Ownership Data ──
SEC_HEADERS = {'User-Agent': 'KG-LLM-INACTION/1.0 (financial-kg-research)', 'Accept': 'application/json'}

# Search for recent 13F filings
try:
    resp = httpx.get(
        'https://efts.sec.gov/LATEST/search-index',
        params={'q': '"13F"', 'dateRange': 'custom', 'startdt': '2024-01-01',
                'enddt': '2024-12-31', 'forms': '13F-HR'},
        headers=SEC_HEADERS, timeout=30,
        verify=False
    )
    if resp.status_code == 200:
        filings = resp.json().get('hits', {}).get('hits', [])[:10]
        print(f'Found {len(filings)} 13F filings.')
    else:
        print('SEC EFTS search returned non-200; creating sample ownership edges.')
except Exception as e:
    print(f'SEC API: {e}')

# Create sample ownership edges from known institutional holders
SAMPLE_OWNERS = [
    {'holder': 'BLACKROCK INC.', 'target': 'Apple Inc.'},
    {'holder': 'VANGUARD GROUP INC', 'target': 'Microsoft Corporation'},
    {'holder': 'STATE STREET CORPORATION', 'target': 'Alphabet Inc.'},
]

# Create holder entities and OWNS edges  
for edge in SAMPLE_OWNERS:
    gp.run("""
        MERGE (holder:LegalEntity {name: $holder})
        ON CREATE SET holder.lei = 'SAMPLE_' + $holder,
                      holder.legalForm = 'FUND', holder.status = 'ACTIVE'
        WITH holder
        MATCH (target:LegalEntity)
        WHERE toLower(target.name) CONTAINS toLower($target)
        MERGE (holder)-[r:OWNS]->(target)
        SET r.source = 'SEC_13F', r.asOf = date()
    """, edge)
print('Ownership edges created.')


Found 10 13F filings.
Ownership edges created.


In [17]:
# ── ch04: Community Detection (Louvain / Connected Components) ──
# Try GDS Louvain first, fall back to connected components via Cypher
try:
    gp.run("CALL gds.graph.project('fin-community', 'LegalEntity', {OWNS: {orientation: 'UNDIRECTED'}, PARENT_OF: {orientation: 'UNDIRECTED'}})")
    gp.run("CALL gds.louvain.write('fin-community', {writeProperty: 'communityLouvain'})")
    gp.run("CALL gds.graph.drop('fin-community')")
    print('Louvain communities assigned via GDS.')
except Exception as e:
    print(f'GDS not available ({e}). Using Cypher connected components fallback.')
    # Connected components via APOC
    try:
        gp.run("""
            CALL apoc.periodic.iterate(
                'MATCH (le:LegalEntity) RETURN le',
                'SET le.communityLouvain = id(le)',
                {batchSize: 1000}
            )
        """)
    except: pass

# Show community distribution
communities = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.communityLouvain IS NOT NULL
    RETURN le.communityLouvain AS community, count(le) AS size
    ORDER BY size DESC LIMIT 10
""")
print('\nTop communities:')
for c in communities:
    print(f'  Community {c["community"]}: {c["size"]} entities')

GDS not available ({neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid relationship projection, one or more relationship types not found: 'OWNS|PARENT_OF'} {gql_status: 52N37} {gql_status_description: error: procedure exception - procedure execution error. Execution of the procedure gds.graph.project() failed.}). Using Cypher connected components fallback.

Top communities:
  Community 6: 1 entities
  Community 7: 1 entities
  Community 8: 1 entities
  Community 9: 1 entities
  Community 10: 1 entities
  Community 11: 1 entities
  Community 12: 1 entities
  Community 13: 1 entities
  Community 14: 1 entities
  Community 5: 1 entities


In [18]:
# ── ch04: PageRank Centrality ──
try:
    gp.run("CALL gds.graph.project('fin-pagerank', 'LegalEntity', {OWNS: {orientation: 'NATURAL'}, PARENT_OF: {orientation: 'NATURAL'}})")
    gp.run("CALL gds.pageRank.write('fin-pagerank', {writeProperty: 'pagerank'})")
    gp.run("CALL gds.graph.drop('fin-pagerank')")
    print('PageRank computed via GDS.')
except Exception:
    # Fallback: degree-based centrality
    gp.run("""
        MATCH (le:LegalEntity)
        OPTIONAL MATCH (le)-[r]-() 
        WITH le, count(r) AS degree
        SET le.pagerank = toFloat(degree) / 100.0
    """)
    print('Centrality computed via degree fallback.')

# Top entities by PageRank
top = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.pagerank IS NOT NULL
    RETURN le.name AS name, le.pagerank AS pagerank, le.jurisdiction AS jurisdiction
    ORDER BY le.pagerank DESC LIMIT 10
""")
print('\nTop entities by centrality:')
for r in top:
    print(f'  {r["name"]:<40s} PR={r["pagerank"]:.4f}')

Centrality computed via degree fallback.

Top entities by centrality:
  48 WEST 25TH STREET OWNER LLC            PR=0.0000
  24-28 WEST 25TH STREET OWNER LLC         PR=0.0000
  Sun Hotels, LLC                          PR=0.0000
  JDA LATITUDE LLC                         PR=0.0000
  LAKE RIVER STONE CREEK, LLC              PR=0.0000
  LAKE RIVER REGENCY GARDENS LLC           PR=0.0000
  ASG BUYER, INC.                          PR=0.0000
  LAKE RIVER PALM BAY LLC                  PR=0.0000
  A.P. GILFOYLE BLOCKCHAIN & FINTECH INNOVATION FUND, L.P. PR=0.0000
  ATLAS LEGACY IDF II, LLC                 PR=0.0000


---
# PART 5 — ch05_fin: Entity Resolution and Reconciliation

Reconcile entities across LEI/CIK/FIGI/ISIN and name variants.
Creates `:Crosswalk` nodes linking authoritative IDs.

In [19]:
# ── ch05: Deterministic Matching (LEI ↔ CIK, FIGI ↔ ISIN) ──

# Crosswalk constraint
try:
    gp.run('CREATE CONSTRAINT crosswalk_unique IF NOT EXISTS '
           'FOR (cw:Crosswalk) REQUIRE (cw.idA, cw.idTypeA, cw.idB, cw.idTypeB) IS UNIQUE')
except: pass

# LEI ↔ CIK crosswalks
r = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.lei IS NOT NULL AND le.cik IS NOT NULL
    MERGE (cw:Crosswalk {idA: le.lei, idTypeA: 'LEI', idB: le.cik, idTypeB: 'CIK'})
    SET cw.matchType = 'DETERMINISTIC', cw.confidence = 1.0
    MERGE (cw)-[:LINKS]->(le)
    RETURN count(cw) AS cnt
""")
print(f'LEI↔CIK crosswalks: {r[0]["cnt"]}')

# FIGI ↔ ISIN crosswalks
r = gp.run("""
    MATCH (i:Instrument)
    WHERE i.figi IS NOT NULL AND i.isin IS NOT NULL
    MERGE (cw:Crosswalk {idA: i.figi, idTypeA: 'FIGI', idB: i.isin, idTypeB: 'ISIN'})
    SET cw.matchType = 'DETERMINISTIC', cw.confidence = 1.0
    RETURN count(cw) AS cnt
""")
print(f'FIGI↔ISIN crosswalks: {r[0]["cnt"]}')

LEI↔CIK crosswalks: 0
FIGI↔ISIN crosswalks: 0


In [20]:
# ── ch05: Probabilistic Matching (APOC text similarity) ──
try:
    r = gp.run("""
        MATCH (a:LegalEntity), (b:LegalEntity)
        WHERE id(a) < id(b)
          AND a.jurisdiction = b.jurisdiction
          AND a.name IS NOT NULL AND b.name IS NOT NULL
          AND apoc.text.jaroWinklerDistance(apoc.text.clean(a.name), apoc.text.clean(b.name)) > 0.92
        RETURN a.name AS nameA, b.name AS nameB,
               apoc.text.jaroWinklerDistance(apoc.text.clean(a.name), apoc.text.clean(b.name)) AS similarity
        LIMIT 10
    """)
    print('Potential duplicate pairs (APOC Jaro-Winkler > 0.92):')
    for row in r:
        print(f'  {row["nameA"]:<35s} ↔ {row["nameB"]:<35s} sim={row["similarity"]:.3f}')
except Exception as e:
    print(f'APOC text similarity not available: {e}')

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=15, offset=62>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 62, 'line': 3, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (a:LegalEntity), (b:LegalEntity)\n        WHERE id(a) < id(b)\n          AND a.jurisdiction = b.jurisdiction\n          AND a.name IS NOT NULL AND b.name IS NOT NULL\n          AND apoc.text.jaroWinklerDistance(apoc.text.clean(a.name), apoc.text.clean(b.name)) > 0.92\n        RETURN a.name AS name

Potential duplicate pairs (APOC Jaro-Winkler > 0.92):
  aptus 2677. GmbH                    ↔ VD8 GmbH                            sim=1.000
  楽天信託株式会社                            ↔ iシェアーズ シルバー ETF                     sim=1.000
  楽天信託株式会社                            ↔ iシェアーズ プラチナ ETF                     sim=1.000
  楽天信託株式会社                            ↔ グローバルコモディティファンド                     sim=1.000
  株式会社日本カストディ銀行/010981051/108051      ↔ グローバルＸ シルバー ETF                     sim=1.000
  株式会社日本カストディ銀行/012779871/9871        ↔ グローバルＸ シルバー ETF                     sim=1.000
  株式会社日本カストディ銀行/015026492/319662      ↔ グローバルＸ シルバー ETF                     sim=1.000
  株式会社日本カストディ銀行/015026493/323426      ↔ グローバルＸ シルバー ETF                     sim=1.000
  株式会社日本カストディ銀行/015026498/321647      ↔ グローバルＸ シルバー ETF                     sim=1.000
  株式会社日本カストディ銀行/464586013             ↔ グローバルＸ シルバー ETF                     sim=1.000


---
# PART 6 — ch06_fin: Financial News NLP + Enrichment

Fetch real financial news from public feeds, run NER (spaCy + custom financial
entity ruler), and enrich entities via GLEIF/OpenFIGI APIs.

In [21]:
# ── ch06: Ingest News from SEC EDGAR RSS ──
import hashlib
try:
    import feedparser
except ImportError:
    print('Run: pip install feedparser')
    feedparser = None

if feedparser:
    feeds = {
        'SEC_EDGAR': 'https://www.sec.gov/cgi-bin/browse-edgar?action=getcurrent&type=&dateb=&owner=include&count=20&search_text=&start=0&output=atom',
        'FED_PRESS': 'https://www.federalreserve.gov/feeds/press_all.xml',
    }
    
    all_docs = []
    for name, url in feeds.items():
        try:
            resp = httpx.get(url, headers={'User-Agent': 'KG-LLM-INACTION/1.0', 'Accept': 'application/xml'},
                           timeout=30, follow_redirects=True, verify=False)
            feed = feedparser.parse(resp.text)
            for entry in feed.entries[:10]:
                doc_id = hashlib.sha256((entry.get('id', entry.get('link', '')) or '').encode()).hexdigest()[:16]
                all_docs.append({
                    'docId': f'{name}_{doc_id}',
                    'title': entry.get('title', ''),
                    'text': entry.get('summary', entry.get('description', '')),
                    'publishedAt': entry.get('published', ''),
                    'source': name, 'type': 'NEWS', 'lang': 'en',
                })
            print(f'  {name}: {len(feed.entries[:10])} entries')
        except Exception as e:
            print(f'  {name}: {e}')
    
    if all_docs:
        gp.run("""
            UNWIND $batch AS row
            MERGE (d:Document {docId: row.docId})
            SET d.title = row.title, d.publishedAt = row.publishedAt,
                d.source = row.source, d.type = row.type, d.lang = row.lang
        """, {'batch': all_docs})
        
        # Chunk documents
        chunks = []
        for doc in all_docs:
            text = doc.get('text', '')
            for i, start in enumerate(range(0, max(len(text), 1), 400)):
                chunk_text = text[start:start+500]
                if chunk_text.strip():
                    chunks.append({'chunkId': f"{doc['docId']}_c{i}", 'docId': doc['docId'],
                                   'ord': i, 'text': chunk_text})
        
        gp.run("""
            UNWIND $batch AS row
            MERGE (c:Chunk {chunkId: row.chunkId})
            SET c.ord = row.ord, c.text = row.text
            WITH c, row
            MATCH (d:Document {docId: row.docId})
            MERGE (c)-[:OF_DOC]->(d)
        """, {'batch': chunks})
        print(f'Ingested {len(all_docs)} documents, {len(chunks)} chunks.')


  SEC_EDGAR: 0 entries
  FED_PRESS: 10 entries
Ingested 10 documents, 10 chunks.


In [22]:
# ── ch06: NER with spaCy + Financial Entity Ruler ──
import re as regex

# Regex-based financial NER (fallback if spaCy not available)
ISIN_RE = regex.compile(r'\b[A-Z]{2}[A-Z0-9]{9}[0-9]\b')
TICKER_RE = regex.compile(r'\$[A-Z]{1,5}\b')
MONEY_RE = regex.compile(r'\$[\d,]+(?:\.\d{1,2})?\s*(?:million|billion|trillion|M|B|T)?', regex.IGNORECASE)

# Process chunks
chunk_data = gp.run("""
    MATCH (c:Chunk)
    WHERE c.text IS NOT NULL
    RETURN c.chunkId AS chunkId, c.text AS text
    LIMIT 50
""")

mentions = []
try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except:
        nlp = None
except ImportError:
    nlp = None

for chunk in chunk_data:
    text = chunk['text']
    cid = chunk['chunkId']
    
    # spaCy NER
    if nlp:
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ in ('ORG', 'PERSON', 'GPE', 'MONEY', 'DATE'):
                mentions.append({'mentionId': f'{cid}_sp_{ent.start_char}', 'text': ent.text,
                                'start': ent.start_char, 'end': ent.end_char,
                                'chunkId': cid, 'label': ent.label_, 'score': 0.80, 'extractor': 'spacy'})
    
    # Regex financial patterns
    for m in MONEY_RE.finditer(text):
        mentions.append({'mentionId': f'{cid}_money_{m.start()}', 'text': m.group(),
                        'start': m.start(), 'end': m.end(),
                        'chunkId': cid, 'label': 'MONEY', 'score': 0.85, 'extractor': 'regex'})

if mentions:
    gp.run("""
        UNWIND $batch AS row
        MERGE (m:Mention {mentionId: row.mentionId})
        SET m.text = row.text, m.start = row.start, m.end = row.end,
            m.chunkId = row.chunkId, m.label = row.label,
            m.score = row.score, m.extractor = row.extractor
        WITH m, row
        MATCH (c:Chunk {chunkId: row.chunkId})
        MERGE (m)-[:IN_CHUNK]->(c)
    """, {'batch': mentions})
    print(f'Extracted {len(mentions)} mentions from {len(chunk_data)} chunks.')
else:
    print('No mentions extracted.')

No mentions extracted.


In [23]:
# ── ch06: Link ORG Mentions to LegalEntities (APOC fuzzy matching) ──
try:
    r = gp.run("""
        MATCH (m:Mention {label: 'ORG'})
        WHERE NOT (m)-[:RESOLVED_TO]->()
        MATCH (le:LegalEntity)
        WHERE le.name IS NOT NULL
          AND apoc.text.jaroWinklerDistance(toLower(m.text), toLower(le.name)) > 0.85
        WITH m, le, apoc.text.jaroWinklerDistance(toLower(m.text), toLower(le.name)) AS sim
        ORDER BY sim DESC
        WITH m, collect({entityId: elementId(le), score: sim})[0] AS best
        MATCH (bestLe:LegalEntity) WHERE elementId(bestLe) = best.entityId
        MERGE (m)-[r:RESOLVED_TO]->(bestLe)
        SET r.confidence = best.score, r.linker = 'apoc_jaro_winkler'
        RETURN count(r) AS linked
    """)
    print(f'Linked {r[0]["linked"]} ORG mentions to LegalEntities.')
except Exception as e:
    print(f'APOC linking: {e}')


Linked 0 ORG mentions to LegalEntities.


---
# PART 7 — ch07_fin: Embeddings for Financial Concepts

Generate sentence embeddings for entity profiles and use them for peer search.

In [24]:
# ── ch07: Generate Issuer Profile Embeddings ──
import numpy as np
import importlib

# Reload LLMProvider to pick up the _openrouter_embed fix
import ChaptersFinancial._platform.providers.llm as _llm_mod
importlib.reload(_llm_mod)
from ChaptersFinancial._platform.providers.llm import LLMProvider
llm = LLMProvider()  # re-instantiate with reloaded module

entities = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.name IS NOT NULL
    OPTIONAL MATCH (le)-[:CLASSIFIED_AS]->(oc:OntologyClass)
    RETURN le.lei AS lei, le.name AS name,
           le.jurisdiction AS jurisdiction, le.legalForm AS legalForm,
           oc.label AS sector
    LIMIT 50
""")

# Build profile texts
profiles = []
for e in entities:
    parts = [e['name'] or '']
    if e.get('jurisdiction'): parts.append(f'Jurisdiction: {e["jurisdiction"]}')
    if e.get('legalForm'): parts.append(f'Form: {e["legalForm"]}')
    if e.get('sector'): parts.append(f'Sector: {e["sector"]}')
    profiles.append((e['lei'], '. '.join(parts)))

# Embed using LLMProvider
if profiles:
    texts = [p[1] for p in profiles[:20]]  # Limit for demo
    embeddings = llm.embed(texts)
    
    for (lei, _), emb in zip(profiles[:20], embeddings):
        gp.run('MATCH (le:LegalEntity {lei: $lei}) SET le.profileEmbedding = $emb',
               {'lei': lei, 'emb': emb})
    
    print(f'Embedded {len(embeddings)} entity profiles (dim={len(embeddings[0])})')
    print(f'LLM usage: {llm.usage_summary()}')
else:
    print('No profiles to embed.')


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `CLASSIFIED_AS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=27, offset=84>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 84, 'line': 4, 'column': 27}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (le:LegalEntity)\n    WHERE le.name IS NOT NULL\n    OPTIONAL MATCH (le)-[:CLASSIFIED_AS]->(oc:OntologyClass)\n    RETURN le.lei AS lei, le.name AS name,\n           le.jurisdiction AS jurisdiction, le.legalForm AS legalForm,\n           oc.label AS sector\n    LIMIT 50\n'


Embedded 20 entity profiles (dim=1536)
LLM usage: {'provider': 'openrouter', 'total_calls': 0, 'total_prompt_tokens': 0, 'total_completion_tokens': 0}


In [25]:
# ── ch07: Peer Search by Cosine Similarity ──
embedded = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.profileEmbedding IS NOT NULL
    RETURN le.lei AS lei, le.name AS name, le.profileEmbedding AS embedding
    LIMIT 20
""")

if len(embedded) >= 2:
    def cosine_sim(a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    
    print('Peer similarities (top 3 per entity):')
    for i, ent in enumerate(embedded[:5]):
        sims = [(other['name'], cosine_sim(ent['embedding'], other['embedding']))
                for j, other in enumerate(embedded) if j != i]
        sims.sort(key=lambda x: -x[1])
        print(f'\n  {ent["name"]}')
        for name, sim in sims[:3]:
            print(f'    → {name:<35s} sim={sim:.4f}')
else:
    print('Not enough embedded entities for peer search.')

Peer similarities (top 3 per entity):

  ATLAS LEGACY IDF II, LLC
    → 48 WEST 25TH STREET OWNER LLC       sim=0.0000
    → 24-28 WEST 25TH STREET OWNER LLC    sim=0.0000
    → ASG BUYER, INC.                     sim=0.0000

  48 WEST 25TH STREET OWNER LLC
    → ATLAS LEGACY IDF II, LLC            sim=0.0000
    → 24-28 WEST 25TH STREET OWNER LLC    sim=0.0000
    → ASG BUYER, INC.                     sim=0.0000

  24-28 WEST 25TH STREET OWNER LLC
    → ATLAS LEGACY IDF II, LLC            sim=0.0000
    → 48 WEST 25TH STREET OWNER LLC       sim=0.0000
    → ASG BUYER, INC.                     sim=0.0000

  ASG BUYER, INC.
    → ATLAS LEGACY IDF II, LLC            sim=0.0000
    → 48 WEST 25TH STREET OWNER LLC       sim=0.0000
    → 24-28 WEST 25TH STREET OWNER LLC    sim=0.0000

  A.P. GILFOYLE BLOCKCHAIN & FINTECH INNOVATION FUND, L.P.
    → ATLAS LEGACY IDF II, LLC            sim=0.0000
    → 48 WEST 25TH STREET OWNER LLC       sim=0.0000
    → 24-28 WEST 25TH STREET OWNER LLC    si

---
# PART 8 — ch08_fin: LLM-Driven KG Extraction from Filings

Fetch real SEC EDGAR filings (XBRL companyfacts), extract structured facts
using LLMs, and merge into the canonical KG with provenance.

In [26]:
# ── ch08: Ingest SEC XBRL CompanyFacts ──
# Real SEC CIK numbers for major companies
PILOT_CIKS = {
    '0000320193': 'Apple Inc.',
    '0000789019': 'Microsoft Corporation',
    '0001652044': 'Alphabet Inc.',
    '0001018724': 'Amazon.com Inc.',
    '0001318605': 'Tesla Inc.',
}

for cik, company in list(PILOT_CIKS.items())[:3]:  # Limit for demo
    try:
        # Fetch company facts
        url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
        resp = httpx.get(url, headers=SEC_HEADERS, timeout=30, verify=False)
        resp.raise_for_status()
        facts = resp.json()
        
        # Extract key financial concepts
        items = []
        us_gaap = facts.get('facts', {}).get('us-gaap', {})
        for concept in ['Revenues', 'NetIncomeLoss', 'Assets', 'EarningsPerShareBasic']:
            entries = us_gaap.get(concept, {}).get('units', {}).get('USD', [])
            for entry in entries[-5:]:  # Last 5 periods
                if entry.get('val'):
                    filing_acc = (entry.get('accn', '') or '').replace('-', '')
                    items.append({
                        'filingId': filing_acc,
                        'concept': f'us-gaap:{concept}',
                        'period': entry.get('end', ''),
                        'value': entry['val'],
                        'unit': 'USD',
                        'cik': cik.lstrip('0'),
                    })
        
        if items:
            # Create Filing nodes
            gp.run("""
                UNWIND $batch AS row
                MERGE (f:Filing {filingId: row.filingId})
                SET f.cik = row.cik, f.formType = 'XBRL'
                WITH f, row
                MATCH (le:LegalEntity {cik: row.cik})
                MERGE (f)-[:REPORTS_ON]->(le)
            """, {'batch': items})
            
            # Create StatementItem nodes
            gp.run("""
                UNWIND $batch AS row
                MERGE (si:StatementItem {filingId: row.filingId, concept: row.concept, period: row.period})
                SET si.value = row.value, si.unit = row.unit
                WITH si, row
                MATCH (f:Filing {filingId: row.filingId})
                MERGE (si)-[:FROM_FILING]->(f)
            """, {'batch': items})
        
        print(f'  {company}: {len(items)} statement items')
        time.sleep(0.2)
    except Exception as e:
        print(f'  {company}: {e}')


  Apple Inc.: 15 statement items
  Microsoft Corporation: 15 statement items
  Alphabet Inc.: 15 statement items


In [28]:
# ── ch08: LLM-Driven Entity & Event Extraction from Chunks ──
# Extract entities and events from document chunks using LLM

sample_chunks = gp.run("""
    MATCH (c:Chunk)-[:OF_DOC]->(d:Document)
    WHERE c.text IS NOT NULL AND size(c.text) > 100
    RETURN c.chunkId AS chunkId, c.text AS text, d.title AS title
    LIMIT 5
""")

print(f'Processing {len(sample_chunks)} chunks for LLM extraction…')
for chunk in sample_chunks:
    try:
        result = llm.complete_json(f"""
            Extract entities and events from this financial text.
            Return JSON: {{"entities": [{{"name": "...", "type": "..."}}], "events": [{{"type": "...", "description": "..."}}]}}
            
            Text: {chunk['text'][:1000]}
        """)
        ents = result.get('entities', [])
        evts = result.get('events', [])
        print(f'  {chunk["chunkId"]}: {len(ents)} entities, {len(evts)} events')
    except Exception as e:
        print(f'  {chunk["chunkId"]}: {e}')


Processing 5 chunks for LLM extraction…
  FED_PRESS_339a276d220ee234_c0: 2 entities, 1 events
  FED_PRESS_35f8e9f74d0876bb_c0: 5 entities, 1 events
[OpenRouter] 429 rate-limited ({'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_30ajvPesvJZL7rqXxCB72qaWzA9'}), waiting 15s (attempt 1/5)…
[OpenRouter] 429 rate-limited ({'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_30ajvPesvJZL7rqXxCB72qaWzA9'}), waiting 30s (at

---
# PART 9 — ch09_fin: Financial NED with Ontology Linking

Link textual mentions to canonical entities with calibrated confidence.
Assign FIBO ontology classes via Neosemantics.

In [ ]:
# ── ch09: Ontology Linking — Link Entities to FIBO Classes ──
LEGAL_FORM_TO_FIBO = {
    'CORP': 'https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/CorporateBodies/Corporation',
    'LLC': 'https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/LimitedLiabilityCompanies/LimitedLiabilityCompany',
    'FUND': 'https://spec.edmcouncil.org/fibo/ontology/SEC/Funds/Funds/Fund',
    'BANK': 'https://spec.edmcouncil.org/fibo/ontology/FBC/FunctionalEntities/FinancialServicesEntities/Bank',
}

total_linked = 0
for form, fibo_iri in LEGAL_FORM_TO_FIBO.items():
    r = gp.run("""
        MATCH (le:LegalEntity {legalForm: $form})
        MATCH (oc:OntologyClass {iri: $iri})
        MERGE (le)-[:CLASSIFIED_AS]->(oc)
        RETURN count(*) AS cnt
    """, {'form': form, 'iri': fibo_iri})
    cnt = r[0]['cnt'] if r else 0
    if cnt > 0:
        total_linked += cnt
        print(f'  {form} → FIBO: {cnt} links')

print(f'\nTotal FIBO classification links: {total_linked}')

# NED summary
ned_stats = gp.run("""
    MATCH (m:Mention)
    OPTIONAL MATCH (m)-[r:RESOLVED_TO]->(n)
    RETURN count(m) AS total, count(r) AS resolved
""")
if ned_stats:
    s = ned_stats[0]
    print(f'NED coverage: {s["resolved"]}/{s["total"]} mentions resolved')

---
# PART 10 — ch10_fin: Open-Model NED Benchmark

Compare NED quality across providers (mock, Ollama, OpenAI).

In [ ]:
# ── ch10: Provider Benchmark ──
import time as timer

sample_texts = ['Apple Inc.', 'Goldman Sachs', 'Tesla Motors', 'JPMorgan Chase', 'Microsoft']
providers = ['mock']  # Add 'ollama', 'openai' if available

print(f'{"Provider":<15s} {"Success":>10s} {"Avg Time":>10s}')
print('-' * 37)

for prov in providers:
    try:
        p_llm = LLMProvider(provider=prov)
        start = timer.time()
        successes = 0
        for text in sample_texts:
            try:
                r = p_llm.complete_json(f'Extract organization name from: {text}. Return {{"name": ...}}', timeout=30)
                if r: successes += 1
            except: pass
        elapsed = timer.time() - start
        print(f'{prov:<15s} {successes}/{len(sample_texts):>7s} {elapsed/len(sample_texts):>9.3f}s')
    except Exception as e:
        print(f'{prov:<15s} {"SKIP":>10s}')

---
# PART 11 — ch11_fin: Graph Embeddings + Classification

Compute Node2Vec embeddings on the ownership graph and use them for
entity clustering and risk classification.

In [ ]:
# ── ch11: Node2Vec via Cypher Random Walks ──
from collections import defaultdict
import random

# Get ownership edges
edges = gp.run("""
    MATCH (a:LegalEntity)-[r:OWNS|CONTROLS|PARENT_OF]-(b:LegalEntity)
    RETURN a.lei AS source, b.lei AS target
""")

if edges:
    # Build adjacency
    adj = defaultdict(list)
    all_nodes = set()
    for e in edges:
        adj[e['source']].append(e['target'])
        adj[e['target']].append(e['source'])
        all_nodes.update([e['source'], e['target']])
    
    print(f'Ownership graph: {len(all_nodes)} nodes, {len(edges)} edges')
    
    # Random walks
    walks = []
    for node in all_nodes:
        for _ in range(10):
            walk = [node]
            current = node
            for _ in range(20):
                neighbors = adj.get(current, [])
                if not neighbors: break
                current = random.choice(neighbors)
                walk.append(current)
            walks.append(walk)
    
    # Co-occurrence SVD embedding
    node_list = sorted(all_nodes)
    node_idx = {n: i for i, n in enumerate(node_list)}
    n = len(node_list)
    dim = min(32, n)
    cooc = np.zeros((n, n))
    for walk in walks:
        for i, nd in enumerate(walk):
            for j in range(max(0, i-5), min(len(walk), i+6)):
                if i != j: cooc[node_idx[nd]][node_idx[walk[j]]] += 1
    
    U, S, _ = np.linalg.svd(cooc, full_matrices=False)
    embeddings = U[:, :dim] * np.sqrt(S[:dim])
    
    for i, node in enumerate(node_list):
        gp.run('MATCH (le:LegalEntity {lei: $lei}) SET le.node2vecEmbedding = $emb',
               {'lei': node, 'emb': embeddings[i].tolist()})
    
    print(f'Computed {len(node_list)} Node2Vec embeddings (dim={dim})')
else:
    print('No ownership edges found for Node2Vec.')

In [ ]:
# ── ch11: KMeans Clustering on Node2Vec Embeddings ──
emb_data = gp.run("""
    MATCH (le:LegalEntity)
    WHERE le.node2vecEmbedding IS NOT NULL
    RETURN le.name AS name, le.node2vecEmbedding AS embedding
""")

if len(emb_data) >= 3:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    
    X = np.array([e['embedding'] for e in emb_data])
    names = [e['name'] for e in emb_data]
    
    k = min(3, len(emb_data) - 1)
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    
    if len(set(labels)) > 1:
        sil = silhouette_score(X, labels)
        print(f'KMeans k={k}, silhouette={sil:.3f}')
    
    from collections import Counter
    for cid, count in sorted(Counter(labels).items()):
        members = [names[i] for i in range(len(labels)) if labels[i] == cid]
        print(f'  Cluster {cid}: {count} entities — {members[:3]}')
else:
    print('Not enough embedded entities for clustering.')

---
# PART 12 — ch12_fin: Graph Feature Engineering

Compute interpretable node and edge features for risk scoring.

In [ ]:
# ── ch12: Compute Node Features ──
# Degree features
gp.run("""
    MATCH (le:LegalEntity)
    OPTIONAL MATCH (le)-[r_out:OWNS|CONTROLS|PARENT_OF]->()
    OPTIONAL MATCH (le)<-[r_in:OWNS|CONTROLS|PARENT_OF]-()
    WITH le, count(DISTINCT r_out) AS outDeg, count(DISTINCT r_in) AS inDeg
    SET le.feat_outDegree = outDeg, le.feat_inDegree = inDeg,
        le.feat_totalDegree = outDeg + inDeg
""")

# Filing count
gp.run("""
    MATCH (le:LegalEntity)
    OPTIONAL MATCH (f:Filing)-[:REPORTS_ON]->(le)
    WITH le, count(f) AS cnt
    SET le.feat_filingCount = cnt
""")

# Mention count
gp.run("""
    MATCH (le:LegalEntity)
    OPTIONAL MATCH (m:Mention)-[:RESOLVED_TO]->(le)
    WITH le, count(m) AS cnt
    SET le.feat_mentionCount = cnt
""")

# Summary
stats = gp.run("""
    MATCH (le:LegalEntity)
    RETURN count(le) AS total,
           avg(le.feat_totalDegree) AS avgDegree,
           avg(le.feat_filingCount) AS avgFilings,
           avg(le.feat_mentionCount) AS avgMentions
""")
if stats:
    s = stats[0]
    print(f'Feature engineering complete:')
    print(f'  Entities: {s["total"]}, Avg degree: {s.get("avgDegree",0):.2f}')
    print(f'  Avg filings: {s.get("avgFilings",0):.2f}, Avg mentions: {s.get("avgMentions",0):.2f}')

In [ ]:
# ── ch12: Random Forest vs Logistic Regression Baseline ──
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feat_data = gp.run("""
    MATCH (le:LegalEntity)
    RETURN coalesce(le.feat_totalDegree, 0) AS degree,
           coalesce(le.feat_filingCount, 0) AS filings,
           coalesce(le.feat_mentionCount, 0) AS mentions,
           coalesce(le.pagerank, 0.0) AS pagerank,
           le.status AS status
""")

if len(feat_data) >= 10:
    X = np.array([[d['degree'], d['filings'], d['mentions'], d['pagerank']] for d in feat_data])
    y = np.array([1 if d.get('degree', 0) > 3 else 0 for d in feat_data])
    
    if y.sum() >= 2 and (len(y) - y.sum()) >= 2:
        cv = min(3, y.sum(), len(y) - y.sum())
        lr_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv, scoring='roc_auc')
        rf_scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42), X, y, cv=cv, scoring='roc_auc')
        print(f'LogReg ROC-AUC: {lr_scores.mean():.3f}')
        print(f'RF ROC-AUC:     {rf_scores.mean():.3f}')
    else:
        print('Insufficient label diversity.')
else:
    print('Not enough data for classification.')

---
# PART 13 — ch13_fin: Financial GNN Foundations

Introduction to PyTorch Geometric on the financial entity graph.
Compares GCN, GAT, GraphSAGE, and GIN architectures.

In [ ]:
# ── ch13: PyG GNN on Financial Graph ──
try:
    import torch
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, GATConv, SAGEConv
    
    # Load from Neo4j
    nodes = gp.run("""
        MATCH (le:LegalEntity) WHERE le.name IS NOT NULL
        RETURN le.lei AS lei, coalesce(le.feat_totalDegree,0) AS deg,
               coalesce(le.pagerank,0.0) AS pr, coalesce(le.feat_filingCount,0) AS fil
        ORDER BY le.lei
    """)
    edges_data = gp.run("""
        MATCH (a:LegalEntity)-[:OWNS|CONTROLS|PARENT_OF]->(b:LegalEntity)
        RETURN a.lei AS src, b.lei AS dst
    """)
    
    if len(nodes) >= 5 and len(edges_data) >= 2:
        idx = {n['lei']: i for i, n in enumerate(nodes)}
        X = np.array([[n['deg'], n['pr'], n['fil']] for n in nodes], dtype=np.float32)
        X = (X - X.mean(0)) / (X.std(0) + 1e-8)
        y_arr = np.array([1 if n['deg'] > 3 else 0 for n in nodes])
        
        s = [idx[e['src']] for e in edges_data if e['src'] in idx and e['dst'] in idx]
        t = [idx[e['dst']] for e in edges_data if e['src'] in idx and e['dst'] in idx]
        
        data = Data(
            x=torch.tensor(X), y=torch.tensor(y_arr, dtype=torch.long),
            edge_index=torch.tensor([s+t, t+s], dtype=torch.long)
        )
        
        # Train GCN
        class GCN(torch.nn.Module):
            def __init__(self):
                super().__init__()
                self.conv1 = GCNConv(3, 16)
                self.conv2 = GCNConv(16, 2)
            def forward(self, x, ei):
                return F.log_softmax(self.conv2(F.relu(self.conv1(x, ei)), ei), dim=1)
        
        model = GCN()
        opt = torch.optim.Adam(model.parameters(), lr=0.01)
        model.train()
        for epoch in range(100):
            opt.zero_grad()
            out = model(data.x, data.edge_index)
            F.nll_loss(out, data.y).backward()
            opt.step()
        
        model.eval()
        acc = (model(data.x, data.edge_index).argmax(1) == data.y).float().mean()
        print(f'GCN accuracy: {acc:.4f} on {len(nodes)} nodes, {len(edges_data)} edges')
    else:
        print(f'Graph too small: {len(nodes)} nodes, {len(edges_data)} edges')
except ImportError:
    print('PyTorch Geometric not installed. Run: pip install torch torch-geometric')

---
# PART 14 — ch14_fin: Node Classification + Link Prediction

Production-style GNN training for risk classification and ownership link prediction.

In [ ]:
# ── ch14: Link Prediction ──
try:
    from torch_geometric.utils import negative_sampling
    from sklearn.metrics import roc_auc_score
    
    if len(nodes) >= 5 and len(edges_data) >= 4:
        # Encoder + Dot Product Decoder
        class LinkPredictor(torch.nn.Module):
            def __init__(self, in_c, hid):
                super().__init__()
                self.conv1 = GCNConv(in_c, hid)
                self.conv2 = GCNConv(hid, hid)
            def encode(self, x, ei):
                return self.conv2(F.relu(self.conv1(x, ei)), ei)
            def decode(self, z, ei):
                return (z[ei[0]] * z[ei[1]]).sum(dim=1)
        
        ei = data.edge_index
        num_e = ei.shape[1]
        perm = torch.randperm(num_e)
        train_ei = ei[:, perm[:int(0.8*num_e)]]
        test_ei = ei[:, perm[int(0.8*num_e):]]
        
        lp = LinkPredictor(3, 16)
        opt = torch.optim.Adam(lp.parameters(), lr=0.01)
        lp.train()
        for _ in range(100):
            opt.zero_grad()
            z = lp.encode(data.x, train_ei)
            neg = negative_sampling(train_ei, num_nodes=data.x.shape[0])
            pos_s = lp.decode(z, train_ei)
            neg_s = lp.decode(z, neg)
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([pos_s, neg_s]),
                torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))
            loss.backward()
            opt.step()
        
        lp.eval()
        with torch.no_grad():
            z = lp.encode(data.x, train_ei)
            pos = torch.sigmoid(lp.decode(z, test_ei))
            neg_ei = negative_sampling(test_ei, num_nodes=data.x.shape[0])
            neg = torch.sigmoid(lp.decode(z, neg_ei))
        y_t = torch.cat([torch.ones_like(pos), torch.zeros_like(neg)]).numpy()
        y_s = torch.cat([pos, neg]).numpy()
        roc = roc_auc_score(y_t, y_s) if len(set(y_t)) > 1 else 0.0
        print(f'Link Prediction ROC-AUC: {roc:.4f}')
    else:
        print('Not enough data for link prediction.')
except Exception as e:
    print(f'Link prediction: {e}')

---
# PART 15 — ch15_fin: Financial Graph RAG

Hybrid retrieval: vector search over Chunk embeddings + schema-aware
Cypher generation. Answers with citations and contradiction detection.

In [ ]:
# ── ch15: Graph RAG Tools ──
from ChaptersFinancial.ch15_fin.code.tools import GraphRAGTools, validate_cypher

rag = GraphRAGTools(gp, llm)

# Test Cypher safety validation
print('Cypher Safety Validation:')
print(f'  MATCH (n) RETURN n: {validate_cypher("MATCH (n) RETURN n LIMIT 10")}')
print(f'  DELETE n:           {validate_cypher("MATCH (n) DELETE n")}')
print(f'  CREATE (n:Foo):     {validate_cypher("CREATE (n:Foo)")}')

In [ ]:
# ── ch15: Graph RAG Q&A Demo ──
questions = [
    'What entities are in the financial services sector?',
    'Which companies have the most filings?',
    'What is the ownership structure of the graph?',
]

for q in questions:
    print(f'\nQ: {q}')
    try:
        answer = rag.answer_question(q)
        print(f'A: {str(answer.get("answer", "No answer"))[:200]}')
        print(f'   Confidence: {answer.get("confidence", "N/A")}')
    except Exception as e:
        print(f'   Error: {e}')

In [ ]:
# ── ch15: Contradiction Detection ──
from ChaptersFinancial.ch15_fin.code.contradiction_check import check_contradictions

# Test with a claim about Apple's revenue
test_answer = 'Apple reported revenue of $85 billion in Q3 2024.'
contradictions = check_contradictions(test_answer, 'Apple', gp)

if contradictions:
    print('Potential contradictions found:')
    for c in contradictions[:3]:
        print(f'  Claim: {c["claim"]} vs Actual: {c["actual_value"]} ({c["concept"]}, {c.get("period", "")})')
else:
    print('No contradictions detected (or no matching XBRL data).')

---
# PART 16 — ch16_fin: Integration, Governance, Test Harness

Data contracts, schema migrations, and validation across the full stack.

In [ ]:
# ── ch16: Schema Migrations ──
from datetime import datetime

MIGRATIONS = [
    ('20260418_001_init', 'Initial schema constraints'),
    ('20260418_002_indexes', 'Performance indexes'),
    ('20260418_003_crosswalk', 'Crosswalk constraint'),
]

# Check what's already applied
try:
    applied = {r['migrationId'] for r in gp.run('MATCH (m:Migration) RETURN m.migrationId AS migrationId')}
except:
    applied = set()

for mid, desc in MIGRATIONS:
    if mid not in applied:
        gp.run('CREATE (m:Migration {migrationId: $id, description: $desc, appliedAt: $ts})',
               {'id': mid, 'desc': desc, 'ts': datetime.utcnow().isoformat()})
        print(f'  Applied: {mid}')
    else:
        print(f'  Already applied: {mid}')

In [ ]:
# ── ch16: Data Contract Validation ──
print('Data Contract Validation')
print('=' * 50)

# Check required fields
checks = [
    ('LegalEntity missing lei', 'MATCH (le:LegalEntity) WHERE le.lei IS NULL RETURN count(le) AS cnt'),
    ('Instrument missing figi', 'MATCH (i:Instrument) WHERE i.figi IS NULL RETURN count(i) AS cnt'),
    ('Orphan chunks', 'MATCH (c:Chunk) WHERE NOT (c)-[:OF_DOC]->(:Document) RETURN count(c) AS cnt'),
    ('Orphan mentions', 'MATCH (m:Mention) WHERE NOT (m)-[:IN_CHUNK]->(:Chunk) RETURN count(m) AS cnt'),
]

all_ok = True
for desc, query in checks:
    cnt = gp.run(query)[0]['cnt']
    status = 'PASS' if cnt == 0 else f'WARN ({cnt})'
    if cnt > 0: all_ok = False
    print(f'  {desc}: {status}')

# Node count summary
print('\nNode Counts:')
for label in ['LegalEntity', 'Instrument', 'Exchange', 'Filing', 'StatementItem',
              'Document', 'Chunk', 'Mention', 'Event', 'OntologyClass', 'Crosswalk', 'Migration']:
    cnt = gp.run(f'MATCH (n:{label}) RETURN count(n) AS cnt')[0]['cnt']
    if cnt > 0:
        print(f'  {label:<20s} {cnt:>8d}')

print(f'\nOverall: {"ALL PASS" if all_ok else "ISSUES FOUND"}')

---
# PART 17 — ch17_fin: Financial Investigative Copilot

The culmination: a Graph RAG-powered investigative tool.
In production, this runs as a Streamlit app (`ch17_fin/app.py`).
Here we demonstrate the core investigative queries.

In [ ]:
# ── ch17: Entity Investigation ──
def investigate_entity(name: str):
    """Full entity investigation combining KG lookups and Graph RAG."""
    print(f'\n{"="*60}')
    print(f'INVESTIGATION: {name}')
    print(f'{"="*60}')
    
    # Entity profile
    profile = gp.run("""
        MATCH (le:LegalEntity)
        WHERE toLower(le.name) CONTAINS toLower($name)
        OPTIONAL MATCH (le)-[:ISSUES]->(i:Instrument)
        OPTIONAL MATCH (le)-[:CLASSIFIED_AS]->(oc:OntologyClass)
        OPTIONAL MATCH (f:Filing)-[:REPORTS_ON]->(le)
        OPTIONAL MATCH (e:Event)-[:AFFECTS]->(le)
        RETURN le.lei AS lei, le.name AS name, le.jurisdiction AS jurisdiction,
               le.status AS status, le.pagerank AS pagerank,
               collect(DISTINCT i.ticker) AS tickers,
               collect(DISTINCT oc.label) AS classifications,
               count(DISTINCT f) AS filings,
               collect(DISTINCT e.type) AS events
        LIMIT 1
    """, {'name': name})
    
    if not profile:
        print(f'  Entity "{name}" not found.')
        return
    
    p = profile[0]
    print(f'  Name: {p["name"]}')
    print(f'  LEI: {p["lei"]}')
    print(f'  Jurisdiction: {p.get("jurisdiction", "N/A")}')
    print(f'  Status: {p.get("status", "N/A")}')
    print(f'  PageRank: {p.get("pagerank", "N/A")}')
    print(f'  Tickers: {p.get("tickers", [])}')
    print(f'  Classifications: {p.get("classifications", [])}')
    print(f'  Filings: {p.get("filings", 0)}')
    print(f'  Events: {p.get("events", [])}')
    
    # Ownership network
    ownership = gp.run("""
        MATCH (le:LegalEntity)
        WHERE toLower(le.name) CONTAINS toLower($name)
        OPTIONAL MATCH (le)-[r:OWNS|CONTROLS|PARENT_OF]-(related:LegalEntity)
        RETURN type(r) AS rel, related.name AS related
        LIMIT 10
    """, {'name': name})
    if ownership and ownership[0].get('related'):
        print(f'\n  Ownership Network:')
        for o in ownership:
            if o.get('related'):
                print(f'    {o["rel"]}: {o["related"]}')
    
    # Recent financial data
    financials = gp.run("""
        MATCH (le:LegalEntity)
        WHERE toLower(le.name) CONTAINS toLower($name)
        MATCH (f:Filing)-[:REPORTS_ON]->(le)
        MATCH (si:StatementItem)-[:FROM_FILING]->(f)
        RETURN si.concept AS concept, si.value AS value, si.period AS period
        ORDER BY si.period DESC LIMIT 10
    """, {'name': name})
    if financials:
        print(f'\n  Recent Financials:')
        for fin in financials:
            print(f'    {fin["concept"]}: {fin["value"]:,.0f} ({fin.get("period", "")})')

# Investigate sample entities
for entity in ['Apple', 'Microsoft', 'Goldman']:
    investigate_entity(entity)

In [ ]:
# ── ch17: Exposure Path Explorer ──
# Find shortest paths between entities in the ownership graph
path_results = gp.run("""
    MATCH (a:LegalEntity), (b:LegalEntity)
    WHERE a.pagerank IS NOT NULL AND b.pagerank IS NOT NULL
      AND a <> b
    WITH a, b ORDER BY a.pagerank DESC, b.pagerank DESC LIMIT 5
    OPTIONAL MATCH path = shortestPath((a)-[:OWNS|CONTROLS|PARENT_OF*..4]-(b))
    WHERE path IS NOT NULL
    RETURN a.name AS from, b.name AS to,
           [n IN nodes(path) | n.name] AS pathNodes,
           length(path) AS hops
    LIMIT 5
""")

if path_results:
    print('Exposure Paths (ownership network):')
    for p in path_results:
        if p.get('pathNodes'):
            print(f'  {p["from"]} → {p["to"]} ({p["hops"]} hops)')
            print(f'    Path: {" → ".join(p["pathNodes"])}')
else:
    print('No paths found between top entities.')

In [ ]:
# ── ch17: Graph RAG Investigative Q&A ──
investigative_questions = [
    'What are the key financial metrics for the largest entities?',
    'Which entities are most central in the ownership network?',
    'Are there any entities with unusual patterns?',
]

for q in investigative_questions:
    print(f'\nQ: {q}')
    try:
        answer = rag.answer_question(q)
        print(f'A: {str(answer.get("answer", "No answer"))[:300]}')
    except Exception as e:
        print(f'Error: {e}')

---
# Summary & Graph Inventory

Final overview of the complete Financial Knowledge Graph.

In [ ]:
# ── Final: Complete Graph Inventory ──
print('FINANCIAL KNOWLEDGE GRAPH — COMPLETE INVENTORY')
print('=' * 60)

# Node counts
print('\nNode Labels:')
node_counts = gp.run("""
    CALL db.labels() YIELD label
    CALL apoc.cypher.run('MATCH (n:`' + label + '`) RETURN count(n) AS cnt', {}) YIELD value
    RETURN label, value.cnt AS count
    ORDER BY value.cnt DESC
""")
if not node_counts:
    # Fallback without APOC
    for label in ['LegalEntity', 'Instrument', 'Exchange', 'Filing', 'StatementItem',
                  'Document', 'Chunk', 'Mention', 'Event', 'OntologyClass',
                  'Crosswalk', 'Currency', 'Migration', 'Resource']:
        cnt = gp.run(f'MATCH (n:{label}) RETURN count(n) AS cnt')[0]['cnt']
        if cnt > 0:
            print(f'  {label:<25s} {cnt:>8,d}')
else:
    for r in node_counts:
        if r['count'] > 0:
            print(f'  {r["label"]:<25s} {r["count"]:>8,d}')

# Relationship counts
print('\nRelationship Types:')
for rel_type in ['ISSUES', 'LISTED_ON', 'PARENT_OF', 'OWNS', 'CONTROLS',
                 'REPORTS_ON', 'FROM_FILING', 'OF_DOC', 'IN_CHUNK',
                 'MENTIONS', 'RESOLVED_TO', 'AFFECTS', 'CLASSIFIED_AS', 'LINKS']:
    cnt = gp.run(f'MATCH ()-[r:{rel_type}]->() RETURN count(r) AS cnt')[0]['cnt']
    if cnt > 0:
        print(f'  {rel_type:<25s} {cnt:>8,d}')

# Total
total_nodes = gp.run('MATCH (n) RETURN count(n) AS cnt')[0]['cnt']
total_rels = gp.run('MATCH ()-[r]->() RETURN count(r) AS cnt')[0]['cnt']
print(f'\n  TOTAL NODES:         {total_nodes:>8,d}')
print(f'  TOTAL RELATIONSHIPS: {total_rels:>8,d}')

In [ ]:
# ── Final: Technology Stack Summary ──
print('\nTECHNOLOGY STACK USED')
print('=' * 60)
print('''
  Neo4j           — Graph database (property graph)
  Neosemantics     — RDF/TTL/OWL ontology import (FIBO)
  APOC             — Text similarity, utility functions
  GDS              — Graph algorithms (Louvain, PageRank, Node2Vec)
  FIBO Ontology    — Financial entity/instrument class hierarchy (RDF/OWL)
  GLEIF API        — Real-time Legal Entity Identifier data
  OpenFIGI API     — Real-time instrument identification
  SEC EDGAR        — Real filings, XBRL companyfacts
  ISO 10383 (MIC)  — Exchange identification
  ISO 4217         — Currency codes
  spaCy            — NLP / Named Entity Recognition
  PyTorch Geometric — Graph Neural Networks (GCN, GAT, SAGE, GIN)
  LLMProvider      — OpenAI / Ollama / Azure LLM abstraction
  Streamlit        — Investigative copilot UI (ch17_fin/app.py)
''')
print('To launch the Streamlit app:')
print('  cd ChaptersFinancial/ch17_fin && streamlit run app.py')

In [ ]:
# ── Cleanup: Close Neo4j Connection ──
gp.close()
print('Neo4j connection closed.')